# Módulo 5 · Clase 5 — Autenticación

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | ¿Por qué autenticar? Tipos de autenticación |
| 2 | API keys — implementación simple |
| 3 | Hashing de contraseñas con `bcrypt` |
| 4 | JWT — qué son y cómo funcionan |
| 5 | OAuth2 Password Flow con FastAPI |
| 6 | Roles y permisos |
| 7 | Proteger endpoints con `Depends` |

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'python-jose[cryptography]', 'passlib[bcrypt]', 'python-multipart', 'requests', 'sqlalchemy']:
    try:
        __import__(pkg.split('[')[0].replace('-','_'))
        print(f"{pkg.split('[')[0]}: ya instalado")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: instalado")

---
## 1. Tipos de autenticación

```
AUTENTICACIÓN = verificar que eres quien dices ser
AUTORIZACIÓN  = verificar que puedes hacer lo que intentas

Tipos comunes en APIs:
┌────────────────┬─────────────────────────────────┬────────────────────┐
│ Tipo           │ Cómo funciona                   │ Cuándo usarlo      │
├────────────────┼─────────────────────────────────┼────────────────────┤
│ API Key        │ Header X-API-Key: abc123        │ Apps M2M, scripts  │
│ Basic Auth     │ Authorization: Basic base64(..) │ Admin interno       │
│ Bearer Token   │ Authorization: Bearer <token>   │ La mayoría de APIs │
│ OAuth2         │ Flujo con login + token         │ APIs con usuarios   │
│ JWT            │ Token firmado, auto-contenido   │ Microservicios     │
│ OAuth2+OIDC    │ Delegación externa (Google...)  │ Login social       │
└────────────────┴─────────────────────────────────┴────────────────────┘

En este curso: API Keys + JWT con OAuth2 Password Flow
```

---
## 2. API Keys

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Security
from fastapi.security import APIKeyHeader
import secrets, uvicorn, threading, time, requests

app_key = FastAPI(title="API con API Keys")

# Almacén de claves (en producción: base de datos)
API_KEYS = {
    "sk_live_abc123": {"nombre": "App móvil",    "permisos": ["read"]},
    "sk_live_xyz789": {"nombre": "Dashboard",    "permisos": ["read", "write"]},
    "sk_live_admin0": {"nombre": "Admin script", "permisos": ["read", "write", "delete"]},
}

# Esquema: busca el header 'X-API-Key'
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def verificar_api_key(api_key: str = Security(api_key_header)):
    if not api_key or api_key not in API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="API key inválida o ausente",
            headers={"WWW-Authenticate": "ApiKey"}
        )
    return API_KEYS[api_key]

def requiere_permiso(permiso: str):
    def verificar(cliente=Depends(verificar_api_key)):
        if permiso not in cliente["permisos"]:
            raise HTTPException(status_code=403, detail=f"Permiso '{permiso}' requerido")
        return cliente
    return verificar


@app_key.get("/datos")  # solo lectura
def leer(cliente=Depends(verificar_api_key)):
    return {"datos": [1, 2, 3], "acceso": cliente["nombre"]}

@app_key.post("/datos")  # necesita 'write'
def escribir(cliente=Depends(requiere_permiso("write"))):
    return {"mensaje": f"Escrito por: {cliente['nombre']}"}

@app_key.delete("/datos")  # necesita 'delete'
def borrar(cliente=Depends(requiere_permiso("delete"))):
    return {"mensaje": "Eliminado"}

@app_key.get("/generar-key")  # genera una nueva clave aleatoria
def generar():
    return {"key": f"sk_live_{secrets.token_urlsafe(16)}"}


cfg = uvicorn.Config(app_key, host="127.0.0.1", port=8020, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B = "http://127.0.0.1:8020"

print("Sin clave (401):")
r = requests.get(f"{B}/datos")
print(f"  {r.status_code}:", r.json())

print("\nClave de solo lectura — GET (200):")
r = requests.get(f"{B}/datos", headers={"X-API-Key": "sk_live_abc123"})
print(f"  {r.status_code}:", r.json())

print("\nClave de solo lectura — POST (403):")
r = requests.post(f"{B}/datos", headers={"X-API-Key": "sk_live_abc123"})
print(f"  {r.status_code}:", r.json())

print("\nClave admin — DELETE (200):")
r = requests.delete(f"{B}/datos", headers={"X-API-Key": "sk_live_admin0"})
print(f"  {r.status_code}:", r.json())

---
## 3. Hashing de contraseñas

**Nunca** guardes contraseñas en texto plano. Usa un hash irreversible con sal:

```
contraseña_texto → [bcrypt/argon2] → hash en BD

Al verificar:
contraseña_texto + hash_en_BD → [bcrypt.verify] → True/False

Características de bcrypt:
✓ Lento a propósito (dificulta ataques de fuerza bruta)
✓ Sal aleatoria incluida en el hash
✓ Factor de coste ajustable
✗ No uses MD5/SHA1 para contraseñas (rápidos, sin sal)
```

In [ ]:
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verificar_password(password: str, hashed: str) -> bool:
    return pwd_context.verify(password, hashed)


contraseña = "miPassword123!"
hashed     = hash_password(contraseña)

print(f"Contraseña original:  {contraseña}")
print(f"Hash bcrypt:          {hashed[:30]}...")  # 60 chars total
print(f"¿Son iguales? (text)  {contraseña == hashed}")
print(f"¿Verificación? (bcrypt) {verificar_password(contraseña, hashed)}")
print(f"¿Contraseña errónea?   {verificar_password('otraPassword', hashed)}")

# Dos hashes de la misma contraseña son DISTINTOS (sal aleatoria)
h2 = hash_password(contraseña)
print(f"\nMismo texto, hash distinto: {hashed[:20]!r} vs {h2[:20]!r}")
print(f"Pero verificar sigue funcionando: {verificar_password(contraseña, h2)}")

---
## 4. JSON Web Tokens (JWT)

```
Un JWT tiene 3 partes separadas por puntos:

eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9   ← HEADER  (base64)
.eyJzdWIiOiJ1c3VhcmlvMSIsImV4cCI6MTY5...  ← PAYLOAD (base64)
.SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQ   ← SIGNATURE

PAYLOAD contiene claims:
{
  "sub":  "usuario1",    ← subject (quién es)
  "exp":  1700000000,    ← expiry (cuándo expira)
  "roles": ["admin"],    ← datos propios
  "iat":  1699996400     ← issued at
}

El servidor firma con su clave secreta.
Cualquiera puede LEER el payload (no es secreto).
Nadie puede MODIFICAR sin invalidar la firma.

IMPORTANTE: No guardes datos sensibles en el payload.
```

In [ ]:
from jose import jwt, JWTError
from datetime import datetime, timedelta, timezone

CLAVE_SECRETA = "super-secreto-que-solo-conoce-el-servidor-min-32-chars"
ALGORITMO     = "HS256"
EXPIRACION_MINUTOS = 30

def crear_token(datos: dict, expira_en: int = EXPIRACION_MINUTOS) -> str:
    payload = datos.copy()
    payload["exp"] = datetime.now(timezone.utc) + timedelta(minutes=expira_en)
    return jwt.encode(payload, CLAVE_SECRETA, algorithm=ALGORITMO)

def decodificar_token(token: str) -> dict:
    try:
        return jwt.decode(token, CLAVE_SECRETA, algorithms=[ALGORITMO])
    except JWTError as e:
        raise HTTPException(status_code=401, detail=f"Token inválido: {e}")


# Crear token
token = crear_token({"sub": "usuario42", "nombre": "Ana García", "rol": "admin"})
print("Token JWT:")
print(token)
print(f"\nLongitud: {len(token)} chars")
print(f"Partes:   {len(token.split('.'))} (header.payload.signature)")

# Decodificar
payload = decodificar_token(token)
print("\nPayload decodificado:")
for k, v in payload.items():
    if k == 'exp':
        print(f"  {k}: {datetime.fromtimestamp(v).strftime('%H:%M:%S')} (expira en ~30 min)")
    else:
        print(f"  {k}: {v}")

# Token manipulado → error
print("\nToken manipulado:")
try:
    decodificar_token(token[:-5] + "XXXXX")
except HTTPException as e:
    print(f"  Error (esperado): {e.detail[:50]}")

---
## 5. OAuth2 Password Flow + JWT con FastAPI

```
FLUJO:

1. Cliente → POST /token  {username, password}
2. Servidor → verifica credenciales → genera JWT → devuelve {access_token}
3. Cliente → GET /me  Authorization: Bearer <JWT>
4. Servidor → decodifica JWT → identifica usuario → devuelve datos

El servidor no guarda sesiones → stateless → escalable
```

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, ConfigDict
from typing import Optional

app_auth = FastAPI(title="API con JWT")

# ── Modelos ──────────────────────────────────────────────────
class Usuario(BaseModel):
    username: str
    nombre:   str
    email:    str
    rol:      str = "usuario"
    activo:   bool = True

class Token(BaseModel):
    access_token: str
    token_type:   str = "bearer"

# ── BD de usuarios (en prod: tabla en SQLite) ─────────────────
USUARIOS_DB = {
    "ana": {
        "username": "ana",
        "nombre":   "Ana García",
        "email":    "ana@empresa.com",
        "rol":      "admin",
        "activo":   True,
        "password_hash": hash_password("secretoAna"),
    },
    "carlos": {
        "username": "carlos",
        "nombre":   "Carlos López",
        "email":    "carlos@empresa.com",
        "rol":      "usuario",
        "activo":   True,
        "password_hash": hash_password("secretoCarlos"),
    },
    "inactivo": {
        "username": "inactivo",
        "nombre":   "Usuario Deshabilitado",
        "email":    "off@empresa.com",
        "rol":      "usuario",
        "activo":   False,
        "password_hash": hash_password("clave123"),
    },
}

# ── OAuth2 scheme ─────────────────────────────────────────────
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

def autenticar_usuario(username: str, password: str) -> Optional[dict]:
    usuario = USUARIOS_DB.get(username)
    if not usuario:
        return None
    if not verificar_password(password, usuario["password_hash"]):
        return None
    return usuario

def obtener_usuario_actual(token: str = Depends(oauth2_scheme)) -> Usuario:
    payload   = decodificar_token(token)
    username  = payload.get("sub")
    if not username or username not in USUARIOS_DB:
        raise HTTPException(status_code=401, detail="Token inválido")
    datos = USUARIOS_DB[username]
    if not datos["activo"]:
        raise HTTPException(status_code=403, detail="Cuenta desactivada")
    return Usuario(**{k: v for k, v in datos.items() if k != "password_hash"})

def solo_admin(usuario: Usuario = Depends(obtener_usuario_actual)) -> Usuario:
    if usuario.rol != "admin":
        raise HTTPException(status_code=403, detail="Se requiere rol admin")
    return usuario


# ── Endpoints ─────────────────────────────────────────────────
@app_auth.post("/token", response_model=Token)
def login(form: OAuth2PasswordRequestForm = Depends()):
    usuario = autenticar_usuario(form.username, form.password)
    if not usuario:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Usuario o contraseña incorrectos",
            headers={"WWW-Authenticate": "Bearer"}
        )
    if not usuario["activo"]:
        raise HTTPException(status_code=403, detail="Cuenta desactivada")

    token = crear_token({"sub": usuario["username"], "rol": usuario["rol"]})
    return Token(access_token=token)


@app_auth.get("/me", response_model=Usuario)
def mi_perfil(usuario: Usuario = Depends(obtener_usuario_actual)):
    return usuario

@app_auth.get("/recursos")
def recursos_publicos(usuario: Usuario = Depends(obtener_usuario_actual)):
    return {"mensaje": f"Hola {usuario.nombre}, aquí están tus recursos"}

@app_auth.get("/admin/usuarios")
def listar_usuarios(admin: Usuario = Depends(solo_admin)):
    return [
        {k: v for k, v in u.items() if k != "password_hash"}
        for u in USUARIOS_DB.values()
    ]


cfg = uvicorn.Config(app_auth, host="127.0.0.1", port=8021, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B = "http://127.0.0.1:8021"

print("=== Login Ana (admin) ===")
r = requests.post(f"{B}/token", data={"username": "ana", "password": "secretoAna"})
token_ana = r.json()["access_token"]
print(f"  Token: {token_ana[:40]}...")

print("\n=== Login contraseña errónea ===")
r = requests.post(f"{B}/token", data={"username": "ana", "password": "mal"})
print(f"  {r.status_code}:", r.json()["detail"])

print("\n=== GET /me ===")
r = requests.get(f"{B}/me", headers={"Authorization": f"Bearer {token_ana}"})
print(r.json())

print("\n=== Admin ve usuarios ===")
r = requests.get(f"{B}/admin/usuarios", headers={"Authorization": f"Bearer {token_ana}"})
print(f"  {r.status_code}: {[u['username'] for u in r.json()]}")

print("\n=== Carlos (no admin) intenta /admin/usuarios ===")
r_carlos = requests.post(f"{B}/token", data={"username": "carlos", "password": "secretoCarlos"})
token_carlos = r_carlos.json()["access_token"]
r = requests.get(f"{B}/admin/usuarios", headers={"Authorization": f"Bearer {token_carlos}"})
print(f"  {r.status_code}:", r.json()["detail"])

---
## 6. Registro de usuarios

In [ ]:
# Endpoint de registro — se añadiría a la app anterior

from pydantic import BaseModel, Field, EmailStr

class RegistroRequest(BaseModel):
    username: str  = Field(min_length=3, max_length=50, pattern=r"^[a-z0-9_]+$")
    nombre:   str  = Field(min_length=1)
    email:    str  = Field(min_length=5)
    password: str  = Field(min_length=8)

@app_auth.post("/registro", response_model=Usuario, status_code=201)
def registrar(datos: RegistroRequest):
    if datos.username in USUARIOS_DB:
        raise HTTPException(status_code=409, detail="El nombre de usuario ya existe")

    if any(u["email"] == datos.email for u in USUARIOS_DB.values()):
        raise HTTPException(status_code=409, detail="El email ya está registrado")

    nuevo = {
        "username":      datos.username,
        "nombre":        datos.nombre,
        "email":         datos.email,
        "rol":           "usuario",
        "activo":        True,
        "password_hash": hash_password(datos.password),
    }
    USUARIOS_DB[datos.username] = nuevo
    return Usuario(**{k: v for k, v in nuevo.items() if k != "password_hash"})


# Probar registro
print("=== Registro nuevo usuario ===")
r = requests.post(f"{B}/registro", json={
    "username": "lucia",
    "nombre":   "Lucía Martínez",
    "email":    "lucia@empresa.com",
    "password": "miClaveSegura99"
})
print(f"  {r.status_code}:", r.json())

print("\n=== Lucía hace login ===")
r = requests.post(f"{B}/token", data={"username": "lucia", "password": "miClaveSegura99"})
print(f"  {r.status_code}: token recibido")

print("\n=== Username duplicado ===")
r = requests.post(f"{B}/registro", json={
    "username": "lucia", "nombre": "Otro", "email": "otro@x.com", "password": "12345678"
})
print(f"  {r.status_code}:", r.json()["detail"])

---
## Práctica

### Ejercicio 1 — Refresh tokens

**Enunciado:** Implementa un sistema de refresh tokens:

- Al hacer login, devuelve un `access_token` (expira en 15 min) y un `refresh_token` (expira en 7 días)
- `POST /token/refresh` — recibe el refresh token y devuelve un nuevo access token
- El refresh token se invalida después de usarse (almacénalo en un set en memoria)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import secrets
from datetime import datetime, timedelta, timezone

# Almacén de refresh tokens activos: {token: username}
refresh_tokens: dict[str, str] = {}

class TokenPair(BaseModel):
    access_token:  str
    refresh_token: str
    token_type:    str = "bearer"

class RefreshRequest(BaseModel):
    refresh_token: str

@app_auth.post("/token/v2", response_model=TokenPair)
def login_v2(form: OAuth2PasswordRequestForm = Depends()):
    usuario = autenticar_usuario(form.username, form.password)
    if not usuario: raise HTTPException(401, "Credenciales incorrectas")

    access  = crear_token({"sub": form.username}, expira_en=15)
    refresh = secrets.token_urlsafe(32)
    refresh_tokens[refresh] = form.username

    return TokenPair(access_token=access, refresh_token=refresh)

@app_auth.post("/token/refresh", response_model=Token)
def renovar(body: RefreshRequest):
    username = refresh_tokens.pop(body.refresh_token, None)  # invalida al usarlo
    if not username:
        raise HTTPException(401, "Refresh token inválido o ya usado")

    usuario = USUARIOS_DB.get(username)
    if not usuario or not usuario["activo"]:
        raise HTTPException(401, "Usuario no válido")

    nuevo_access = crear_token({"sub": username}, expira_en=15)
    return Token(access_token=nuevo_access)
```
</details>

---
### Ejercicio 2 — Cambio de contraseña

**Enunciado:** Añade el endpoint `POST /me/cambiar-password`:

- Requiere autenticación (Bearer token)
- Body: `{"password_actual": "...", "password_nuevo": "...", "confirmacion": "..."}`
- Validar que `password_actual` es correcto
- Validar que `password_nuevo == confirmacion`
- Validar que el nuevo es distinto del actual
- Actualizar el hash en la BD

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
class CambioPassword(BaseModel):
    password_actual: str = Field(min_length=1)
    password_nuevo:  str = Field(min_length=8)
    confirmacion:    str = Field(min_length=8)

@app_auth.post("/me/cambiar-password")
def cambiar_password(
    datos:   CambioPassword,
    usuario: Usuario = Depends(obtener_usuario_actual)
):
    datos_bd = USUARIOS_DB[usuario.username]

    if not verificar_password(datos.password_actual, datos_bd["password_hash"]):
        raise HTTPException(400, "La contraseña actual es incorrecta")

    if datos.password_nuevo != datos.confirmacion:
        raise HTTPException(400, "La nueva contraseña y la confirmación no coinciden")

    if verificar_password(datos.password_nuevo, datos_bd["password_hash"]):
        raise HTTPException(400, "La nueva contraseña debe ser diferente a la actual")

    datos_bd["password_hash"] = hash_password(datos.password_nuevo)
    return {"mensaje": "Contraseña actualizada correctamente"}
```
</details>

---
### Ejercicio 3 — Auditoría

**Enunciado:** Añade un sistema de auditoría que registre los accesos:

- Modelo `LogAcceso`: id, username, endpoint, metodo, timestamp, ip (simulada)
- Registra automáticamente cada request autenticado con un `BackgroundTask`
- `GET /admin/logs` — solo admin, lista los últimos 50 logs

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
from fastapi import BackgroundTasks, Request
from datetime import datetime

logs_acceso: list[dict] = []

def registrar_acceso(username: str, endpoint: str, metodo: str):
    logs_acceso.append({
        "id":        len(logs_acceso) + 1,
        "username":  username,
        "endpoint":  endpoint,
        "metodo":    metodo,
        "timestamp": datetime.now().isoformat(),
        "ip":        "127.0.0.1"  # simulada
    })

@app_auth.get("/recursos-auditados")
def recursos_auditados(
    request:    Request,
    bg_tasks:   BackgroundTasks,
    usuario:    Usuario = Depends(obtener_usuario_actual)
):
    bg_tasks.add_task(registrar_acceso, usuario.username, str(request.url.path), request.method)
    return {"datos": "recursos privados", "usuario": usuario.nombre}

@app_auth.get("/admin/logs")
def ver_logs(admin: Usuario = Depends(solo_admin)):
    return logs_acceso[-50:]
```
</details>

---
### Ejercicio 4 — Proteger la API de biblioteca

**Enunciado:** Toma la API de biblioteca de la Clase 4 y añádele autenticación:

- `GET /libros` y `GET /libros/{id}` — cualquier usuario autenticado
- `POST /libros`, `PATCH /libros/{id}` — solo rol `editor` o `admin`
- `DELETE /libros/{id}` — solo `admin`
- `GET /autores` — público (sin autenticación)
- El campo `creado_por` en Libro debe registrar el username del creador

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
# Dependencias de roles
def rol_editor_o_admin(usuario: Usuario = Depends(obtener_usuario_actual)):
    if usuario.rol not in ("editor", "admin"):
        raise HTTPException(403, "Se requiere rol editor o admin")
    return usuario

def solo_admin_bib(usuario: Usuario = Depends(obtener_usuario_actual)):
    if usuario.rol != "admin":
        raise HTTPException(403, "Solo admin puede eliminar libros")
    return usuario

# GET público (sin Depends de auth)
@app.get("/autores")  # ya existente, sin cambios
def listar_autores_pub(db: Session = Depends(get_db)):
    return db.query(Autor).all()

# GET autenticado
@app.get("/libros/privado", response_model=list[LibroConAutor])
def listar_auth(db: Session = Depends(get_db), _: Usuario = Depends(obtener_usuario_actual)):
    return db.query(Libro).all()

# POST solo editor/admin — guarda creador
@app.post("/libros/auth", response_model=LibroConAutor, status_code=201)
def crear_auth(datos: LibroCreate, db: Session = Depends(get_db),
               usuario: Usuario = Depends(rol_editor_o_admin)):
    libro = Libro(**datos.model_dump(), creado_por=usuario.username)  # añadir Column a modelo ORM
    db.add(libro); db.commit(); db.refresh(libro)
    return libro

# DELETE solo admin
@app.delete("/libros/auth/{id}", status_code=204)
def eliminar_auth(id: int, db: Session = Depends(get_db),
                  _: Usuario = Depends(solo_admin_bib)):
    libro = db.get(Libro, id)
    if not libro: raise HTTPException(404, "No encontrado")
    db.delete(libro); db.commit()
```
</details>

---
### Ejercicio 5 (reto) — API con auth completa

**Enunciado:** Construye una API de gestión de proyectos con autenticación y autorización completa:

**Roles:** `admin`, `manager`, `developer`

**Recursos y permisos:**
- `GET /proyectos` — todos los autenticados
- `POST /proyectos` — manager y admin
- `GET /proyectos/{id}/miembros` — todos
- `POST /proyectos/{id}/miembros` — manager del proyecto o admin
- `DELETE /proyectos/{id}` — solo admin

**Regla clave:** cada proyecto tiene un `manager_username`. Solo ese manager (o admin) puede añadir miembros a su proyecto.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests

app_proj = FastAPI(title="Gestión de Proyectos")

# Usuarios
USERS = {
    "admin1":  {"rol": "admin",     "hash": hash_password("admin1pass")},
    "mgr1":    {"rol": "manager",   "hash": hash_password("mgr1pass")},
    "dev1":    {"rol": "developer", "hash": hash_password("dev1pass")},
}

# Proyectos en memoria
proyectos: dict[int, dict] = {}
miembros:  dict[int, list] = {}   # {proyecto_id: [usernames]}
seq = 0

class ProyectoCreate(BaseModel):
    nombre:      str = Field(min_length=1)
    descripcion: Optional[str] = None

class MiembroAdd(BaseModel):
    username: str

oauth2 = OAuth2PasswordBearer(tokenUrl="/login")

def usuario_actual(token: str = Depends(oauth2)):
    payload  = decodificar_token(token)
    username = payload.get("sub")
    if username not in USERS: raise HTTPException(401, "Token inválido")
    return {"username": username, "rol": USERS[username]["rol"]}

@app_proj.post("/login")
def login_proj(form: OAuth2PasswordRequestForm = Depends()):
    u = USERS.get(form.username)
    if not u or not verificar_password(form.password, u["hash"]):
        raise HTTPException(401, "Credenciales incorrectas")
    return {"access_token": crear_token({"sub": form.username}), "token_type": "bearer"}

@app_proj.get("/proyectos")
def listar(u=Depends(usuario_actual)): return list(proyectos.values())

@app_proj.post("/proyectos", status_code=201)
def crear(datos: ProyectoCreate, u=Depends(usuario_actual)):
    if u["rol"] not in ("manager", "admin"): raise HTTPException(403, "Se requiere manager o admin")
    global seq; seq += 1
    p = {"id": seq, "manager_username": u["username"], **datos.model_dump()}
    proyectos[seq] = p; miembros[seq] = [u["username"]]
    return p

@app_proj.get("/proyectos/{id}/miembros")
def ver_miembros(id: int, u=Depends(usuario_actual)):
    if id not in proyectos: raise HTTPException(404, "Proyecto no encontrado")
    return miembros.get(id, [])

@app_proj.post("/proyectos/{id}/miembros")
def añadir_miembro(id: int, body: MiembroAdd, u=Depends(usuario_actual)):
    if id not in proyectos: raise HTTPException(404, "Proyecto no encontrado")
    p = proyectos[id]
    es_manager_del_proyecto = u["username"] == p["manager_username"]
    if u["rol"] != "admin" and not es_manager_del_proyecto:
        raise HTTPException(403, "Solo el manager del proyecto o admin puede añadir miembros")
    if body.username not in USERS: raise HTTPException(404, "Usuario no existe")
    if body.username not in miembros[id]:
        miembros[id].append(body.username)
    return {"miembros": miembros[id]}

@app_proj.delete("/proyectos/{id}", status_code=204)
def eliminar(id: int, u=Depends(usuario_actual)):
    if u["rol"] != "admin": raise HTTPException(403, "Solo admin puede eliminar")
    if id not in proyectos: raise HTTPException(404, "Proyecto no encontrado")
    del proyectos[id]; del miembros[id]
```
</details>

---
## Resumen

```python
# Hash de contraseña
pwd_context = CryptContext(schemes=["bcrypt"])
hashed = pwd_context.hash("mi_password")
ok     = pwd_context.verify("mi_password", hashed)

# Crear JWT
token = jwt.encode({"sub": username, "exp": datetime.now(tz) + timedelta(minutes=30)},
                   SECRET_KEY, algorithm="HS256")

# Decodificar JWT
payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])

# OAuth2 Password Flow
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

@app.post("/token")
def login(form: OAuth2PasswordRequestForm = Depends()):
    # verificar form.username / form.password
    return {"access_token": crear_token({"sub": form.username}), "token_type": "bearer"}

def usuario_actual(token: str = Depends(oauth2_scheme)):
    payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
    return payload["sub"]

@app.get("/privado")
def ruta_privada(user = Depends(usuario_actual)):  # requiere token válido
    return {"usuario": user}
```

## ¿Qué viene en la próxima clase?

En la **Clase 6** añadimos la capa profesional: documentación OpenAPI/Swagger enriquecida, CORS para frontends, middleware de logging y rate limiting para proteger la API contra abuso.